In [ ]:
!nvidia-smi

In [1]:
from google.colab import drive
drive.mount('/content/drive')

In [2]:
%%capture
import os
if not os.path.exists('/content/drive/MyDrive/[ICLR] Embedding KD/ICLR-MDD-new_idea_2'):
    !unzip /content/drive/MyDrive/[ICLR] Embedding KD/ICLR-MDD-new_idea_2.zip

In [3]:
%cd /content/ICLR-MDD-new_idea_2

In [4]:
# Remove stale cached teacher embeddings before training
!rm -rf cache
!mkdir -p cache
!echo "Removed cache/"

In [ ]:
!pip install -r requirements.txt

In [5]:
# TMKD primary run: full batch kernel with the fixed global coefficient lambda=1.
# The script expects to run from scripts/, so all paths below are relative to that directory.
!test -f data/merged_9_data_3k_each_ver2.csv || (echo 'Missing data/merged_9_data_3k_each_ver2.csv. Re-run the unzip cell or check the archive.' && false)
!mkdir -p checkpoints/tmkd_full_lambda1 analysis/tmkd_full_lambda1
!bash -c 'set -o pipefail; cd scripts && TRAIN_DATA="../data/merged_9_data_3k_each_ver2.csv" STUDENT_MODEL="google-bert/bert-base-uncased" TEACHER_MODEL="Qwen/Qwen3-Embedding-4B" BATCH_SIZE=4 EPOCHS=5 LR=1e-5 MAX_LENGTH=256 SAVE_DIR="../checkpoints/tmkd_full_lambda1" LAMBDA_TMKD=1.0 TMKD_BLOCK_SIZE=512 TMKD_MODE=full bash train_tmkd.sh 2>&1 | tee "../analysis/tmkd_full_lambda1/train.log"'

In [ ]:
# Run validation first, then reuse validation-selected pair thresholds on the test sets.
import re
from pathlib import Path

import torch
from transformers import AutoModel, AutoTokenizer

from src.evaluation.evaluation_automodel import (
    eval_classification_task,
    eval_pair_task,
    eval_sts_task,
    eval_cls_tasks,
    eval_pair_tasks,
    eval_sts_tasks,
    test_cls_tasks,
    test_pair_tasks,
    test_sts_tasks,
)

checkpoint_dir = Path("checkpoints/tmkd_full_lambda1")
best_checkpoint = checkpoint_dir / "best_model.pt"

if best_checkpoint.exists():
    checkpoint_path = best_checkpoint
else:
    checkpoints = sorted(
        checkpoint_dir.glob("checkpoint_epoch_*.pt"),
        key=lambda p: int(re.search(r"checkpoint_epoch_(\d+)", p.stem).group(1)),
    )
    assert checkpoints, f"No checkpoint found in {checkpoint_dir}"
    checkpoint_path = checkpoints[-1]

print(f"Loading checkpoint: {checkpoint_path}")

try:
    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
except TypeError:
    checkpoint = torch.load(checkpoint_path, map_location="cpu")

cfg = checkpoint.get("config", {})
student_model_name = cfg.get("student_model_name", "google-bert/bert-base-uncased")
print(f"Student model: {student_model_name}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = AutoModel.from_pretrained(student_model_name)
tokenizer = AutoTokenizer.from_pretrained(student_model_name, use_fast=True)
model.load_state_dict(checkpoint["model_state_dict"])
model.to(device)
model.eval()

print("\n" + "=" * 80 + "\nValidation benchmarks\n" + "=" * 80)
validation_classification = eval_classification_task(model, eval_cls_tasks, tokenizer)
validation_pair, pair_thresholds = eval_pair_task(model, eval_pair_tasks, tokenizer)
validation_sts = eval_sts_task(model, eval_sts_tasks, tokenizer)

print("\n" + "=" * 80 + "\nTest benchmarks\n" + "=" * 80)
test_classification = eval_classification_task(model, test_cls_tasks, tokenizer)
test_pair, _ = eval_pair_task(model, test_pair_tasks, tokenizer, thresholds=pair_thresholds)
test_sts = eval_sts_task(model, test_sts_tasks, tokenizer)

print("\nAll benchmark evaluations finished.")